# RiskRank pipeline walkthrough

End-to-end: Bronze → Silver → Gold → Models → AdjustedRisk.
Run the producers + consumer first (see README), then execute these cells.

In [ ]:
from riskrank.config import get_settings
from riskrank.spark.session import build_spark_session

settings = get_settings()
spark = build_spark_session(settings, app_name='riskrank-notebook')
spark.sparkContext.setLogLevel('WARN')
spark

## 1. Bronze → Silver

In [ ]:
from riskrank.spark.silver import run_silver
run_silver(settings, trigger_mode='availableNow')

In [ ]:
from riskrank.spark.gold import read_silver_nvd
read_silver_nvd(spark, settings).select(
    'cve_id', 'cvss_vector', 'cvss_base_score', 'attack_vector', 'privileges_required'
).show(5, truncate=False)

## 2. Silver → Gold (time-aware dataset)

In [ ]:
from riskrank.spark.gold import build_gold_observations
gold = build_gold_observations(spark, settings, write=True)
print('rows:', gold.count())
gold.select('cve_id', 'observation_date', 'epss_current', 'kev_within_90_days').show(5)

## 3. Train models + tune weights

In [ ]:
from riskrank.models.train import run_training
report = run_training(settings)
report['test_ranking']

## 4. Score a new CVSS vector

In [ ]:
import json
from pyspark.ml import PipelineModel
from riskrank.paths import ProjectPaths
from riskrank.models.scoring import score_single_vector

paths = ProjectPaths(settings)
w = json.loads((paths.models / 'weights.json').read_text())
model_a = PipelineModel.load(str(paths.models / 'model_a_epss'))
model_b = PipelineModel.load(str(paths.models / 'model_b_kev'))

score_single_vector(
    spark, model_a, model_b,
    cvss_vector='CVSS:3.1/AV:N/AC:H/PR:L/UI:R/S:U/C:H/I:H/A:N',
    cvss_base_score=7.0,
    weights=(w['w1_cvss'], w['w2_exploit'], w['w3_kev']),
)